Proyecto Olist ML
Predicción de retrasos logísticos y segmentación de clientes

Institución: Universidad de Cundinamarca

Programa: Ingeniería de Sistemas

Asignatura: Ciencia de Datos

Integrantes: Juan David Aguilar Páez y Nicolas Mauricio Vasquez

Cuadernillo: 01 — Comprensión de datos


# Comprensión inicial del conjunto de datos Olist

**Proyecto:** Predicción de retrasos logísticos y segmentación de clientes mediante Machine Learning.

## Objetivo general

Analizar la estructura y características iniciales del conjunto de datos Olist para establecer las bases del desarrollo del proyecto.

## Objetivos específicos

1. Identificar los archivos, dimensiones y variables del conjunto de datos.
2. Reconocer las relaciones entre las tablas y las variables relevantes para el problema.
3. Identificar posibles problemas de calidad de datos que deban tratarse durante el preprocesamiento.


In [1]:
import pandas as pd
from pathlib import Path

ruta = Path("../data/raw")
pedidos = pd.read_csv(ruta / "olist_orders_dataset.csv")

pedidos.shape

(99441, 8)

In [2]:
pedidos.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [3]:
pedidos.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [4]:
pedidos.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Estados de los pedidos

In [5]:
pedidos["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [6]:
pd.crosstab(
    pedidos["order_status"],
    pedidos["order_delivered_customer_date"].isna()
)

order_delivered_customer_date,False,True
order_status,,
approved,0,2
canceled,6,619
created,0,5
delivered,96470,8
invoiced,0,314
processing,0,301
shipped,0,1107
unavailable,0,609


In [7]:
pedidos["order_id"].duplicated().sum()


np.int64(0)

In [8]:
archivos = list(Path("../data/raw").glob("*.csv"))

for archivo in sorted(archivos):
    df = pd.read_csv(archivo)
    print(f"{archivo.name}: {df.shape}")

olist_customers_dataset.csv: (99441, 5)
olist_geolocation_dataset.csv: (1000163, 5)
olist_order_items_dataset.csv: (112650, 7)
olist_order_payments_dataset.csv: (103886, 5)
olist_order_reviews_dataset.csv: (99224, 7)
olist_orders_dataset.csv: (99441, 8)
olist_products_dataset.csv: (32951, 9)
olist_sellers_dataset.csv: (3095, 4)
product_category_name_translation.csv: (71, 2)


Estructuras de las tablas

In [9]:
for archivo in sorted(archivos):
    df = pd.read_csv(archivo, nrows=0)
    print(f"\n{archivo.stem}")
    print(df.columns.tolist())


olist_customers_dataset
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

olist_geolocation_dataset
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

olist_order_items_dataset
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

olist_order_payments_dataset
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

olist_order_reviews_dataset
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

olist_orders_dataset
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

olist_products_dataset
['product_id', 'product_category_name', 'product_name_lenght', 'product_

¿Qué significa esto?
La tabla olist_orders_dataset será nuestro punto de partida para la predicción de retrasos.
- customer_id relaciona cada pedido con su registro de cliente.
- order_id conecta pedidos con artículos, pagos y reseñas.
- product_id conecta los artículos comprados con las características de los productos.
- seller_id identifica los vendedores asociados a los artículos.
- customer_unique_id será especialmente importante para la segmentación, porque permite identificar clientes a través de distintas compras.

In [10]:
clientes = pd.read_csv(ruta / "olist_customers_dataset.csv")

pd.DataFrame({
    "Registros": [len(clientes)],
    "ID duplicados": [clientes["customer_id"].duplicated().sum()],
    "Compradores únicos": [clientes["customer_unique_id"].nunique()]
})

,Registros,ID duplicados,Compradores únicos
0,99441,0,96096


In [11]:
pedidos["customer_id"].isin(clientes["customer_id"]).value_counts()

customer_id
True    99441
Name: count, dtype: int64

Compradores recurrentes

In [12]:
compras_cliente = clientes["customer_unique_id"].value_counts()

print("Compradores con un registro:", (compras_cliente == 1).sum())
print("Compradores con varios registros:", (compras_cliente > 1).sum())
print("Máximo de registros por comprador:", compras_cliente.max())

Compradores con un registro: 93099
Compradores con varios registros: 2997
Máximo de registros por comprador: 17


Artículos por pedido

In [13]:
items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

print("Registros de artículos:", len(items))
print("Pedidos distintos:", items["order_id"].nunique())
print("Pedidos con varios artículos:", (items["order_id"].value_counts() > 1).sum())

Registros de artículos: 112650
Pedidos distintos: 98666
Pedidos con varios artículos: 9803


In [14]:
sin_items = pedidos[~pedidos["order_id"].isin(items["order_id"])]

sin_items["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [15]:
entregados = pedidos[pedidos["order_status"] == "delivered"]

entregados[[
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].isna().sum()

order_delivered_customer_date    8
order_estimated_delivery_date    0
dtype: int64

convertir las fechas y calcular los retrasos

In [16]:
entregados = entregados.dropna(subset=[
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]).copy()

entregados["order_delivered_customer_date"] = pd.to_datetime(
    entregados["order_delivered_customer_date"]
)

entregados["order_estimated_delivery_date"] = pd.to_datetime(
    entregados["order_estimated_delivery_date"]
)

In [17]:
entregados["retraso"] = (
    entregados["order_delivered_customer_date"]
    > entregados["order_estimated_delivery_date"]
).astype(int)

entregados["retraso"].value_counts()

retraso
0    88644
1     7826
Name: count, dtype: int64

In [18]:
entregados["retraso_dia"] = (
    entregados["order_delivered_customer_date"].dt.normalize()
    > entregados["order_estimated_delivery_date"].dt.normalize()
).astype(int)

pd.crosstab(entregados["retraso"], entregados["retraso_dia"])

retraso_dia,0,1
retraso,,
0,88644,0
1,1292,6534


In [19]:
entregados = entregados.drop(columns="retraso")
entregados = entregados.rename(columns={"retraso_dia": "retraso"})

Consistencia temporal

In [20]:
fecha_compra = pd.to_datetime(entregados["order_purchase_timestamp"])

(fecha_compra > entregados["order_delivered_customer_date"]).sum()

np.int64(0)

In [21]:
fecha_aprobacion = pd.to_datetime(entregados["order_approved_at"])

(fecha_aprobacion > entregados["order_delivered_customer_date"]).sum()

np.int64(61)

In [22]:
anomalias = entregados[
    fecha_aprobacion > entregados["order_delivered_customer_date"]
]

anomalias[[
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_customer_date"
]].head(10)

,order_purchase_timestamp,order_approved_at,order_delivered_customer_date
199,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-25 23:58:19
483,2018-07-24 11:32:11,2018-07-29 23:30:52,2018-07-27 18:55:57
1986,2018-08-09 17:36:47,2018-08-20 15:55:42,2018-08-17 16:45:45
3659,2018-01-30 09:43:45,2018-02-04 23:31:47,2018-02-01 20:10:38
11738,2017-09-01 18:45:33,2017-09-13 22:04:39,2017-09-11 14:15:02
13470,2018-07-23 08:50:48,2018-07-28 23:31:06,2018-07-26 17:03:43
14562,2017-09-01 19:04:22,2017-09-13 22:06:11,2017-09-08 20:13:03
16345,2018-07-22 13:07:02,2018-07-27 23:30:58,2018-07-25 18:21:18
18097,2018-07-03 10:30:52,2018-07-05 16:40:30,2018-07-04 22:42:41
20557,2018-06-30 18:49:36,2018-07-05 16:25:45,2018-07-05 12:58:37



## Conclusiones

1. El conjunto de datos Olist contiene nueve archivos CSV relacionados, con información sobre pedidos, clientes, productos, vendedores, pagos y entregas. Su estructura permite abordar problemas de predicción logística y segmentación de clientes.

2. La tabla de pedidos contiene 99.441 registros con identificadores únicos. La relación con clientes presenta correspondencia completa, mientras que la tabla de artículos tiene múltiples registros para algunos pedidos, por lo que será necesario considerar su nivel de detalle al integrar los datos.

3. Se identificaron valores faltantes y posibles inconsistencias temporales. De los pedidos entregados, 96.470 cuentan con fechas suficientes para calcular el retraso. Al comparar días del calendario, 6.534 pedidos (6,77 %) fueron clasificados como tardíos, evidenciando un desbalance entre las clases.

4. Se identificaron 96.096 compradores únicos, de los cuales 2.997 aparecen en varios registros. Esta información constituye un punto de partida para estudiar la segmentación de clientes.
